In [0]:
from pyspark.sql.functions import *

df = spark.table("workspace.bronze.bronze_exchange_rates")

# Cleaning the df (Selecting imp. fields only)
cleaned_df = df.select(
    col("currency_code"),
    col("exchange_rate_to_usd"),
    col("effective_date"),
    col("_modified"),
    col("ingestion_ts") 
)

display(cleaned_df)

In [0]:
parsed_date = coalesce(
    try_to_date(col("effective_date"), "MM/dd/yyyy"),
    try_to_date(col("effective_date"), "M/dd/yyyy"),
    try_to_date(col("effective_date"), "MM/d/yyyy"),
    try_to_date(col("effective_date"), "M/d/yyyy"),

    try_to_date(col("effective_date"), "dd/MM/yyyy"),
    try_to_date(col("effective_date"), "d/MM/yyyy"),
    try_to_date(col("effective_date"), "dd/M/yyyy"),
    try_to_date(col("effective_date"), "d/M/yyyy"),

    try_to_date(col("effective_date"), "yyyy/MM/dd"),
    try_to_date(col("effective_date"), "yyyy/M/dd"),
    try_to_date(col("effective_date"), "yyyy/MM/d"),
    try_to_date(col("effective_date"), "yyyy/M/d"),

    try_to_date(col("effective_date"), "MM-dd-yyyy"),
    try_to_date(col("effective_date"), "M-dd-yyyy"),
    try_to_date(col("effective_date"), "MM-d-yyyy"),
    try_to_date(col("effective_date"), "M-d-yyyy"),

    try_to_date(col("effective_date"), "yyyy-MM-dd"),
    try_to_date(col("effective_date"), "yyyy-MM-d"),
    try_to_date(col("effective_date"), "yyyy-M-dd"),
    try_to_date(col("effective_date"), "yyyy-M-d"),

    try_to_date(col("effective_date"), "d-MM-yyyy"),
    try_to_date(col("effective_date"), "dd-M-yyyy"),
    try_to_date(col("effective_date"), "d-M-yyyy")
)

transformed_df = (
    cleaned_df
    .withColumn("parsed_date", parsed_date)
    .withColumn(
        "effective_date",
        when(
            col("parsed_date").isNotNull(),
            date_format(col("parsed_date"), "dd-MM-yyyy")
        ).otherwise(col("effective_date"))
    )
    .drop("parsed_date")
)

display(transformed_df)

In [0]:
transformed_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.silver_exchange_rates")